# 5.23 — Evaluating Regression Models Using MAE

Mean Absolute Error (MAE) answers a simple question:

> On average, how far off are the predictions?

Because MAE is measured in the **same units as the target**, it is usually the easiest regression metric to interpret and communicate.

In this notebook we will:

- compute MAE for a **baseline** (`DummyRegressor(mean)`) vs **Linear Regression**
- interpret MAE relative to the target scale
- use **cross-validation** with MAE (`neg_mean_absolute_error`)

In [2]:
import sys
from pathlib import Path

repo_root = Path.cwd().resolve()
while not (repo_root / "src").exists() and repo_root.parent != repo_root:
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from src.config import (
    ALL_FEATURES,
    CATEGORICAL_FEATURES,
    EXCLUDED_COLUMNS,
    NUMERICAL_FEATURES,
    TARGET_SOURCE_COLUMN,
)
from src.data_loader import load_training_frame
from src.preprocessing import separate_features_and_target, split_data

## 1) Load data and define a regression target

We use `yield_kg` as a continuous target.

In [6]:
df = load_training_frame()

print(df[TARGET_SOURCE_COLUMN].describe())
df.head()

count       4.0000
mean      950.0000
std       208.1666
min       700.0000
25%       850.0000
50%       950.0000
75%      1050.0000
max      1200.0000
Name: yield_kg, dtype: float64


,crop,region,price,yield_kg
0,Rice,Maharashtra,2100,1200
1,Wheat,Karnataka,1950,1000
2,Maize,Gujarat,1800,900
3,Cotton,Telangana,5000,700


## 2) Train/test split (no leakage)

Split first. Any fitting (scalers, encoders, models) must happen **after** the split.

In [4]:
X, y = separate_features_and_target(
    df,
    target_column=TARGET_SOURCE_COLUMN,
    feature_columns=[c for c in ALL_FEATURES if c in df.columns],
    excluded_columns=EXCLUDED_COLUMNS,
)

X_train, X_test, y_train, y_test = split_data(X, y, test_size=0.2, random_state=42)

numeric_cols = [c for c in NUMERICAL_FEATURES if c in X_train.columns]
cat_cols = [c for c in CATEGORICAL_FEATURES if c in X_train.columns]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

X_train_p = preprocessor.fit_transform(X_train)
X_test_p = preprocessor.transform(X_test)

baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train_p, y_train)
baseline_pred = baseline.predict(X_test_p)

model = LinearRegression()
model.fit(X_train_p, y_train)
model_pred = model.predict(X_test_p)

baseline_mae = mean_absolute_error(y_test, baseline_pred)
model_mae = mean_absolute_error(y_test, model_pred)
improvement = baseline_mae - model_mae
pct_improve = (improvement / baseline_mae) * 100 if baseline_mae != 0 else np.nan

mean_target = float(y_test.mean())
mae_pct_of_mean_target = (model_mae / abs(mean_target)) * 100 if mean_target != 0 else np.nan

summary = pd.DataFrame(
    {
        "mae": [baseline_mae, model_mae],
    },
    index=["DummyRegressor(mean)", "LinearRegression"],
)

print(f"Baseline MAE: {baseline_mae:.4f}")
print(f"Model MAE:    {model_mae:.4f}")
print(f"Improvement:  {improvement:.4f} ({pct_improve:.1f}%)")
print(f"Mean target:  {mean_target:.4f}")
print(f"MAE % mean:   {mae_pct_of_mean_target:.1f}%")

summary

Baseline MAE: 66.6667
Model MAE:    1.5653
Improvement:  65.1014 (97.7%)
Mean target:  1000.0000
MAE % mean:   0.2%


,mae
DummyRegressor(mean),66.666667
LinearRegression,1.565279


## 3) Cross-validation with MAE

A single train/test split can be misleading, especially on small datasets.

We can estimate average MAE more reliably with cross-validation.

Note: scikit-learn returns **negative** MAE for `neg_mean_absolute_error` (it uses a “higher is better” convention), so we flip the sign before reporting.

In [5]:
cv_pipeline = Pipeline(
    steps=[
        (
            "preprocess",
            ColumnTransformer(
                transformers=[
                    ("num", StandardScaler(), numeric_cols),
                    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
                ],
                remainder="drop",
                verbose_feature_names_out=False,
            ),
        ),
        ("model", LinearRegression()),
    ]
)

n_splits = min(5, len(X_train))
if n_splits >= 2:
    cv = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    scores = cross_val_score(
        cv_pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring="neg_mean_absolute_error",
    )

    mae_scores = -scores
    print("CV MAE per fold:", np.round(mae_scores, 4))
    print("Mean CV MAE:", float(mae_scores.mean()))
    print("Std CV MAE:", float(mae_scores.std()))
else:
    print("Not enough training samples for cross-validation.")


CV MAE per fold: [1875.      359.375   200.8621]
Mean CV MAE: 811.7456896551724
Std CV MAE: 754.6141988576798
